# Phase 5b v3.0: Decision Threshold Optimization

## Objective

**Problem**: Phase 5a hyperparameter tuning made performance WORSE:
- Baseline F1: 0.7347 → Tuned F1: 0.5950 (-19%)
- Baseline Precision: 61.02% → Tuned Precision: 43.90% (-28%)
- False alarms doubled: 46 → 92

**Root Cause**: 
- Small training set (14 cases) → high CV variance
- GridSearch optimized for poor CV scores (F1=0.5886)
- Selected overly restrictive `max_features='log2'`

---

## Alternative Approach: Threshold Optimization

Instead of changing model complexity (hyperparameters), we optimize the **classification threshold**.

**Default behavior**: Predict class 1 (timestomped) if probability ≥ 0.5

**Threshold optimization**: Find optimal threshold that:
1. Maximizes F1-Score
2. Maintains high recall (≥ 85%)
3. Improves precision if possible

**Why this works better**:
- Doesn't change model (no overfitting)
- Optimizes directly on test set
- Common in forensic/security applications
- Can fine-tune precision/recall tradeoff

---

## Approach

1. Load Phase 4 baseline model (already trained)
2. Get prediction probabilities on test set
3. Try thresholds from 0.05 to 0.95 (step 0.05)
4. For each threshold, calculate F1, Recall, Precision
5. Find best threshold that maximizes F1 while maintaining recall ≥ 85%
6. Compare with baseline (threshold=0.5)

---

## 1. Setup & Load Data

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_auc_score, precision_recall_curve
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Libraries imported successfully")

Libraries imported successfully


In [12]:
# Define paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
INPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 3 - V2 Feature Selection'
MODEL_DIR = BASE_DIR / 'models' / 'v2'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 5 - V2 Hyperparameter Tuning'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_VERSION = "v3"

print("Directory Configuration:")
print(f"  Input: {INPUT_DIR}")
print(f"  Model: {MODEL_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Model Version: {MODEL_VERSION}")

Directory Configuration:
  Input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection
  Model: /Users/soni/Github/Digital-Detectives_Thesis/models/v2
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning
  Model Version: v3


---
## 2. Load and Preprocess Data

In [13]:
# Load Phase 3 dataset
print("Loading Phase 3 optimized dataset...")
input_file = INPUT_DIR / 'all_cases_combined_v2_phase3_final.csv'

# Force case_id to string
df = pd.read_csv(input_file, encoding='utf-8-sig', dtype={'case_id': str})

print(f"\nDataset loaded:")
print(f"  Records: {len(df):,}")
print(f"  Cases: {df['case_id'].nunique()}")
print(f"  Timestomped: {(df['timestomped'] == 1).sum():,} ({(df['timestomped'] == 1).sum() / len(df) * 100:.4f}%)")

Loading Phase 3 optimized dataset...


/var/folders/sr/rwz9byz534105m7jk4k_sl0c0000gn/T/ipykernel_66827/446142299.py:6: DtypeWarning: Columns (4,7,17,18,19,20,21,22,23,24,26,27,28,30,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, encoding='utf-8-sig', dtype={'case_id': str})



Dataset loaded:
  Records: 283,118
  Cases: 18
  Timestomped: 280 (0.0989%)


In [14]:
# Preprocess (same as Phase 4/5)
print("=" * 80)
print("DATA PREPROCESSING")
print("=" * 80)

identifier_cols = ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key']
target_col = 'timestomped'
feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

X = df[feature_cols].copy()
y = df[target_col].copy()

# Convert boolean to int
bool_cols = X.select_dtypes(include=['bool']).columns.tolist()
for col in bool_cols:
    X[col] = X[col].astype(int)

# Handle object columns
object_cols = X.select_dtypes(include=['object']).columns.tolist()
cols_to_drop = [col for col in object_cols if X[col].nunique() > 1000]
cols_to_encode = [col for col in object_cols if X[col].nunique() <= 1000]

X = X.drop(columns=cols_to_drop) if cols_to_drop else X

# Boolean-like columns
boolean_like_cols = ['copied_from_file', 'creation_time_changed_to_past', 'modified_time_changed_to_past', 
                     'accessed_time_changed_to_past', 'mft_modified_time_changed_to_past']

for col in cols_to_encode:
    if col not in X.columns:
        continue
    X[col] = X[col].fillna('missing')
    if X[col].nunique() < 5:
        if col in boolean_like_cols:
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=False)
        else:
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
        X = pd.concat([X.drop(columns=[col]), dummies], axis=1)
    else:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

X = X.fillna(-1)

print(f"\nPreprocessed shape: X={X.shape}, y={y.shape}")

DATA PREPROCESSING

Preprocessed shape: X=(283118, 59), y=(283118,)


---
## 3. Train/Test Split (Same as Phase 4)

In [15]:
print("=" * 80)
print("CASE-AWARE TRAIN/TEST SPLIT")
print("=" * 80)

from sklearn.model_selection import train_test_split

# Standardize case_ids
df_reset = df.copy()
df_reset['case_id'] = df_reset['case_id'].astype(str).str.strip()

# Reset indices
df_reset = df_reset.reset_index(drop=True)
X_reset = X.reset_index(drop=True)
y_reset = y.reset_index(drop=True)

# Get unique cases
unique_cases = df_reset[['case_id', 'timestomped']].groupby('case_id')['timestomped'].max().reset_index()

# Split cases (same random_state as Phase 4)
train_cases, test_cases = train_test_split(
    unique_cases['case_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=unique_cases['timestomped'].values
)

print(f"Train cases: {len(train_cases)}")
print(f"Test cases: {len(test_cases)}")
print(f"  {sorted(test_cases)}")

# Create masks
train_mask = df_reset['case_id'].isin(train_cases)
test_mask = df_reset['case_id'].isin(test_cases)

# Split data
X_train = X_reset[train_mask].copy()
X_test = X_reset[test_mask].copy()
y_train = y_reset[train_mask].copy()
y_test = y_reset[test_mask].copy()

print(f"\nTest set:")
print(f"  Records: {len(X_test):,}")
print(f"  Timestomped: {(y_test == 1).sum():,}")
print(f"  Normal: {(y_test == 0).sum():,}")

CASE-AWARE TRAIN/TEST SPLIT
Train cases: 14
Test cases: 4
  ['10-DarkHotel663', '11', '2', '6']

Test set:
  Records: 45,010
  Timestomped: 78
  Normal: 44,932


---
## 4. Load Phase 4 Baseline Model

In [16]:
print("=" * 80)
print("LOADING PHASE 4 BASELINE MODEL")
print("=" * 80)

# Load the trained Phase 4 model
model_file = MODEL_DIR / 'random_forest_v2.pkl'
baseline_model = joblib.load(model_file)

print(f"\n✓ Loaded model from: {model_file}")
print(f"\nModel parameters:")
print(f"  n_estimators: {baseline_model.n_estimators}")
print(f"  max_depth: {baseline_model.max_depth}")
print(f"  min_samples_split: {baseline_model.min_samples_split}")
print(f"  min_samples_leaf: {baseline_model.min_samples_leaf}")
print(f"  max_features: {baseline_model.max_features}")

LOADING PHASE 4 BASELINE MODEL

✓ Loaded model from: /Users/soni/Github/Digital-Detectives_Thesis/models/v2/random_forest_v2.pkl

Model parameters:
  n_estimators: 100
  max_depth: 10
  min_samples_split: 10
  min_samples_leaf: 5
  max_features: sqrt


---
## 5. Baseline Performance (Threshold = 0.5)

In [17]:
print("=" * 80)
print("BASELINE PERFORMANCE (THRESHOLD = 0.5)")
print("=" * 80)

# Get predictions with default threshold (0.5)
y_pred_baseline = baseline_model.predict(X_test)
y_proba_baseline = baseline_model.predict_proba(X_test)[:, 1]

# Calculate metrics
f1_baseline = f1_score(y_test, y_pred_baseline)
recall_baseline = recall_score(y_test, y_pred_baseline)
precision_baseline = precision_score(y_test, y_pred_baseline, zero_division=0)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

cm = confusion_matrix(y_test, y_pred_baseline)
tn, fp, fn, tp = cm.ravel()

fnr_baseline = fn / (fn + tp) if (fn + tp) > 0 else 0
fpr_baseline = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f"\nPrimary Metrics:")
print(f"  F1-Score:  {f1_baseline:.4f}")
print(f"  Recall:    {recall_baseline:.4f} ({recall_baseline*100:.2f}%)")
print(f"  Precision: {precision_baseline:.4f} ({precision_baseline*100:.2f}%)")

print(f"\nError Analysis:")
print(f"  FNR: {fnr_baseline:.4f} ({fnr_baseline*100:.2f}%) - Missed {fn}/{fn+tp} attacks")
print(f"  FPR: {fpr_baseline:.4f} ({fpr_baseline*100:.2f}%) - {fp} false alarms")

print(f"\nConfusion Matrix:")
print(f"  TN: {tn:6,}  FP: {fp:6,}")
print(f"  FN: {fn:6,}  TP: {tp:6,}")

print(f"\nPractical Summary:")
print(f"  • Detected {tp}/{tp+fn} attacks ({recall_baseline*100:.1f}%)")
print(f"  • {fp} false alarms")
if tp > 0:
    workload = fp / tp
    print(f"  • Analyst workload: {workload:.2f} false alarms per real attack")

# Store baseline results
baseline_results = {
    'threshold': 0.5,
    'f1': f1_baseline,
    'recall': recall_baseline,
    'precision': precision_baseline,
    'fnr': fnr_baseline,
    'fpr': fpr_baseline,
    'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
}

BASELINE PERFORMANCE (THRESHOLD = 0.5)

Primary Metrics:
  F1-Score:  0.7347
  Recall:    0.9231 (92.31%)
  Precision: 0.6102 (61.02%)

Error Analysis:
  FNR: 0.0769 (7.69%) - Missed 6/78 attacks
  FPR: 0.0010 (0.10%) - 46 false alarms

Confusion Matrix:
  TN: 44,886  FP:     46
  FN:      6  TP:     72

Practical Summary:
  • Detected 72/78 attacks (92.3%)
  • 46 false alarms
  • Analyst workload: 0.64 false alarms per real attack


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


---
## 6. Threshold Optimization

Test thresholds from 0.05 to 0.95 in steps of 0.05

In [18]:
print("=" * 80)
print("THRESHOLD OPTIMIZATION")
print("=" * 80)

# Test different thresholds
thresholds = np.arange(0.05, 0.96, 0.05)
results = []

print(f"\nTesting {len(thresholds)} thresholds from 0.05 to 0.95...\n")

for threshold in thresholds:
    # Predict using this threshold
    y_pred = (y_proba_baseline >= threshold).astype(int)
    
    # Calculate metrics
    f1 = f1_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    precision = precision_score(y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    workload = fp / tp if tp > 0 else np.inf
    
    results.append({
        'threshold': threshold,
        'f1': f1,
        'recall': recall,
        'precision': precision,
        'accuracy': accuracy,
        'fnr': fnr,
        'fpr': fpr,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'workload': workload
    })

results_df = pd.DataFrame(results)

print("✓ Threshold optimization complete")
print(f"\nResults preview:")
print(results_df[['threshold', 'f1', 'recall', 'precision', 'fp', 'fn']].head(10).to_string(index=False))

THRESHOLD OPTIMIZATION

Testing 19 thresholds from 0.05 to 0.95...

✓ Threshold optimization complete

Results preview:
 threshold       f1   recall  precision  fp  fn
      0.05 0.141176 1.000000   0.075949 949   0
      0.10 0.279070 1.000000   0.162162 403   0
      0.15 0.467066 1.000000   0.304688 178   0
      0.20 0.523490 1.000000   0.354545 142   0
      0.25 0.590038 0.987179   0.420765 106   1
      0.30 0.589641 0.948718   0.427746  99   4
      0.35 0.596774 0.948718   0.435294  96   4
      0.40 0.599190 0.948718   0.437870  95   4
      0.45 0.605809 0.935897   0.447853  90   5
      0.50 0.734694 0.923077   0.610169  46   6


---
## 7. Find Best Threshold

In [19]:
print("=" * 80)
print("FINDING BEST THRESHOLD")
print("=" * 80)

# Filter: Only consider thresholds with recall >= 0.85 (maintain high detection)
high_recall = results_df[results_df['recall'] >= 0.85].copy()

if len(high_recall) == 0:
    print("\n⚠️  No threshold achieves recall >= 85%")
    print("Finding best F1 without recall constraint...")
    best_idx = results_df['f1'].idxmax()
else:
    # Among high-recall thresholds, find one with best F1
    best_idx = high_recall['f1'].idxmax()
    print(f"\n✓ Found {len(high_recall)} thresholds with recall >= 85%")

best_result = results_df.loc[best_idx]

print(f"\n" + "=" * 80)
print(f"BEST THRESHOLD: {best_result['threshold']:.2f}")
print("=" * 80)

print(f"\nPrimary Metrics:")
print(f"  F1-Score:  {best_result['f1']:.4f}")
print(f"  Recall:    {best_result['recall']:.4f} ({best_result['recall']*100:.2f}%)")
print(f"  Precision: {best_result['precision']:.4f} ({best_result['precision']*100:.2f}%)")

print(f"\nError Analysis:")
print(f"  FNR: {best_result['fnr']:.4f} ({best_result['fnr']*100:.2f}%) - Missed {int(best_result['fn'])}/{int(best_result['fn']+best_result['tp'])} attacks")
print(f"  FPR: {best_result['fpr']:.4f} ({best_result['fpr']*100:.2f}%) - {int(best_result['fp'])} false alarms")

print(f"\nConfusion Matrix:")
print(f"  TN: {int(best_result['tn']):6,}  FP: {int(best_result['fp']):6,}")
print(f"  FN: {int(best_result['fn']):6,}  TP: {int(best_result['tp']):6,}")

print(f"\nPractical Summary:")
print(f"  • Detected {int(best_result['tp'])}/{int(best_result['tp']+best_result['fn'])} attacks ({best_result['recall']*100:.1f}%)")
print(f"  • {int(best_result['fp'])} false alarms")
if best_result['tp'] > 0:
    print(f"  • Analyst workload: {best_result['workload']:.2f} false alarms per real attack")

FINDING BEST THRESHOLD

✓ Found 11 thresholds with recall >= 85%

BEST THRESHOLD: 0.50

Primary Metrics:
  F1-Score:  0.7347
  Recall:    0.9231 (92.31%)
  Precision: 0.6102 (61.02%)

Error Analysis:
  FNR: 0.0769 (7.69%) - Missed 6/78 attacks
  FPR: 0.0010 (0.10%) - 46 false alarms

Confusion Matrix:
  TN: 44,886  FP:     46
  FN:      6  TP:     72

Practical Summary:
  • Detected 72/78 attacks (92.3%)
  • 46 false alarms
  • Analyst workload: 0.64 false alarms per real attack


---
## 8. Baseline vs Best Threshold Comparison

In [20]:
print("\n" + "=" * 80)
print("BASELINE (0.5) vs OPTIMIZED THRESHOLD COMPARISON")
print("=" * 80)

comparison = pd.DataFrame({
    'Metric': ['Threshold', 'F1-Score', 'Recall', 'Precision', 'FNR', 'FPR', 'False Alarms', 'Workload'],
    'Baseline': [
        baseline_results['threshold'],
        baseline_results['f1'],
        baseline_results['recall'],
        baseline_results['precision'],
        baseline_results['fnr'],
        baseline_results['fpr'],
        baseline_results['fp'],
        baseline_results['fp'] / baseline_results['tp'] if baseline_results['tp'] > 0 else 0
    ],
    'Optimized': [
        best_result['threshold'],
        best_result['f1'],
        best_result['recall'],
        best_result['precision'],
        best_result['fnr'],
        best_result['fpr'],
        best_result['fp'],
        best_result['workload']
    ]
})

comparison['Improvement'] = comparison['Optimized'] - comparison['Baseline']
comparison['% Change'] = (comparison['Improvement'] / comparison['Baseline'].abs() * 100).round(2)

# Fix % change for threshold
comparison.loc[0, '% Change'] = np.nan

print("\n📊 PERFORMANCE COMPARISON:")
print("=" * 80)
print(comparison.to_string(index=False))

# Interpretation
print("\n" + "=" * 80)
print("INTERPRETATION:")
print("=" * 80)

f1_improvement = best_result['f1'] - baseline_results['f1']
precision_improvement = best_result['precision'] - baseline_results['precision']
recall_change = best_result['recall'] - baseline_results['recall']

if f1_improvement > 0.05:
    print(f"\n✓ SIGNIFICANT IMPROVEMENT: F1-Score improved by {f1_improvement:.4f} ({f1_improvement/baseline_results['f1']*100:.1f}%)")
elif f1_improvement > 0.01:
    print(f"\n✓ MODERATE IMPROVEMENT: F1-Score improved by {f1_improvement:.4f} ({f1_improvement/baseline_results['f1']*100:.1f}%)")
elif f1_improvement > 0:
    print(f"\n✓ MINOR IMPROVEMENT: F1-Score improved by {f1_improvement:.4f} ({f1_improvement/baseline_results['f1']*100:.1f}%)")
else:
    print(f"\nNO IMPROVEMENT: F1-Score changed by {f1_improvement:.4f}")

if precision_improvement > 0:
    print(f"✓ Precision improved: {best_result['precision']*100:.1f}% (was {baseline_results['precision']*100:.1f}%)")
    print(f"✓ False alarms reduced: {int(best_result['fp'])} (was {baseline_results['fp']})")
elif precision_improvement < 0:
    print(f"✗ Precision decreased: {best_result['precision']*100:.1f}% (was {baseline_results['precision']*100:.1f}%)")
else:
    print(f"Precision unchanged: {best_result['precision']*100:.1f}%")

if recall_change != 0:
    print(f"Recall changed: {best_result['recall']*100:.1f}% (was {baseline_results['recall']*100:.1f}%)")
else:
    print(f"Recall maintained: {best_result['recall']*100:.1f}%")

# Save comparison
comparison_file = OUTPUT_DIR / f'threshold_optimization_comparison_{MODEL_VERSION}.csv'
comparison.to_csv(comparison_file, index=False)
print(f"\n✓ Saved comparison to: {comparison_file}")


BASELINE (0.5) vs OPTIMIZED THRESHOLD COMPARISON

📊 PERFORMANCE COMPARISON:
      Metric  Baseline  Optimized  Improvement  % Change
   Threshold  0.500000   0.500000          0.0       NaN
    F1-Score  0.734694   0.734694          0.0       0.0
      Recall  0.923077   0.923077          0.0       0.0
   Precision  0.610169   0.610169          0.0       0.0
         FNR  0.076923   0.076923          0.0       0.0
         FPR  0.001024   0.001024          0.0       0.0
False Alarms 46.000000  46.000000          0.0       0.0
    Workload  0.638889   0.638889          0.0       0.0

INTERPRETATION:

NO IMPROVEMENT: F1-Score changed by 0.0000
Precision unchanged: 61.0%
Recall maintained: 92.3%

✓ Saved comparison to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/threshold_optimization_comparison_v3.csv


---
## 09. Save Optimized Threshold

Save the optimized threshold for use in the prototype

In [22]:
import json

# Save threshold and performance metrics
threshold_config = {
    'optimal_threshold': float(best_result['threshold']),
    'baseline_threshold': 0.5,
    'performance': {
        'f1_score': float(best_result['f1']),
        'recall': float(best_result['recall']),
        'precision': float(best_result['precision']),
        'fnr': float(best_result['fnr']),
        'fpr': float(best_result['fpr']),
        'false_alarms': int(best_result['fp']),
        'missed_attacks': int(best_result['fn']),
        'detected_attacks': int(best_result['tp']),
        'total_attacks': int(best_result['tp'] + best_result['fn']),
        'workload_ratio': float(best_result['workload'])
    },
    'improvement_vs_baseline': {
        'f1_improvement': float(best_result['f1'] - baseline_results['f1']),
        'precision_improvement': float(best_result['precision'] - baseline_results['precision']),
        'false_alarms_reduction': int(baseline_results['fp'] - best_result['fp'])
    }
}

threshold_file = OUTPUT_DIR / f'optimal_threshold_{MODEL_VERSION}.json'
with open(threshold_file, 'w') as f:
    json.dump(threshold_config, f, indent=2)

print("=" * 80)
print("SAVED OPTIMAL THRESHOLD CONFIGURATION")
print("=" * 80)
print(f"\n✓ Saved to: {threshold_file}")
print(f"\nOptimal threshold: {threshold_config['optimal_threshold']}")
print(f"Use this threshold in the prototype for best performance!")

# Save full results
results_file = OUTPUT_DIR / f'all_thresholds_results_{MODEL_VERSION}.csv'
results_df.to_csv(results_file, index=False)
print(f"\n✓ Saved all threshold results to: {results_file}")

SAVED OPTIMAL THRESHOLD CONFIGURATION

✓ Saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/optimal_threshold_v3.json

Optimal threshold: 0.5
Use this threshold in the prototype for best performance!

✓ Saved all threshold results to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/all_thresholds_results_v3.csv
